#**Atividade para Casa**

**Objetivo:** Aplicar todas as etapas de pré-processamento em um dataset real.

**Instruções:**
1. Baixar o notebook exemplo do repositório (dataset Titanic)
2. Executar o código completo (verificar se funciona)
3. Testar as mudanças sugeridas no Slide 14
4. Responder no notebook:
*   Qual estratégia deu o melhor resultado? Por quê?
*   O que acontece se você não escolonar os dados?
*   O que acontece se você não tratar os vetores nulos?
5. Subir o notebook respondido na pasta da semana 12 do repositório.



##**Importação das Bibliotecas e Leitura dos Dados**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

#1. Carregar os dados
# Exibir as primeiras linhas para verificar display(df.head())

##**Modelo Base**

In [ ]:
# Definindo as features (X) e a variável alvo (y)
features = ['Age', 'Sex', 'Pclass', 'Embarked', 'SibSp', 'Parch']
X = df[features]
y = df['Survived']

# Dividir os dados em treino e teste (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- PRÉ-PROCESSAMENTO BASE ---
numeric_features = ['Age', 'Pclass', 'SibSp', 'Parch']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_features = ['Sex', 'Embarked']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Pipeline Final
pipeline_base = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# Treinamento e Avaliação
pipeline_base.fit(X_train, y_train)
y_pred_base = pipeline_base.predict(X_test)
acc_base = accuracy_score(y_test, y_pred_base)

print(f"Acurácia do modelo base (LogisticRegression + StandardScaler + Median): {acc_base:.4f}")

##**Testando  as Mudanças do Slide 14**

In [ ]:
# --- TESTE 1: Regressão Logística + StandardScaler + Média ---
numeric_transformer_t1 = Pipeline(steps=[('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler())])
preprocessor_t1 = ColumnTransformer(transformers=[('num', numeric_transformer_t1, numeric_features), ('cat', categorical_transformer, categorical_features)])
pipeline_t1 = Pipeline(steps=[('preprocessor', preprocessor_t1), ('classifier', LogisticRegression(random_state=42))])
pipeline_t1.fit(X_train, y_train)
acc_t1 = accuracy_score(y_test, pipeline_t1.predict(X_test))

# --- TESTE 2: Regressão Logística + MinMaxScaler + Mediana ---
numeric_transformer_t2 = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', MinMaxScaler())])
preprocessor_t2 = ColumnTransformer(transformers=[('num', numeric_transformer_t2, numeric_features), ('cat', categorical_transformer, categorical_features)])
pipeline_t2 = Pipeline(steps=[('preprocessor', preprocessor_t2), ('classifier', LogisticRegression(random_state=42))])
pipeline_t2.fit(X_train, y_train)
acc_t2 = accuracy_score(y_test, pipeline_t2.predict(X_test))

# --- TESTE 3: Árvore de Decisão + StandardScaler + Mediana ---
pipeline_t3 = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', DecisionTreeClassifier(random_state=42))])
pipeline_t3.fit(X_train, y_train)
acc_t3 = accuracy_score(y_test, pipeline_t3.predict(X_test))

# --- TESTE 4: Testando o que acontece SEM Escalonamento (Regressão Logística) ---
numeric_transformer_sem_scaler = Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]) # Removi o Scaler
preprocessor_sem_scaler = ColumnTransformer(transformers=[('num', numeric_transformer_sem_scaler, numeric_features), ('cat', categorical_transformer, categorical_features)])
pipeline_t4 = Pipeline(steps=[('preprocessor', preprocessor_sem_scaler), ('classifier', LogisticRegression(random_state=42, max_iter=1000))])
pipeline_t4.fit(X_train, y_train)
acc_t4 = accuracy_score(y_test, pipeline_t4.predict(X_test))

print("=== RESULTADOS DOS TESTES ===")
print(f"Base (LogReg + StandardScaler + Median): {acc_base:.4f}")
print(f"T1   (LogReg + StandardScaler + Mean):   {acc_t1:.4f}")
print(f"T2   (LogReg + MinMaxScaler + Median):   {acc_t2:.4f}")
print(f"T3   (DecisionTree + StdScaler + Median):{acc_t3:.4f}")
print(f"T4   (LogReg SEM Escalonamento + Median):{acc_t4:.4f}")

##**Respostas para as Perguntas dos Slides**

###**1. Qual estratégia deu o melhor resultado? Por quê?**

A estratégia Modelo Base (Regressão Logística + StandardScaler + Tratamento de Nulos com Mediana) deu o melhor resultado com os hiperparâmetros padrões, chegando a ~81.0% de acurácia. A troca do Scaler para MinMaxScaler fez o modelo perder um pouco de precisão (caiu para ~79.3%), assim como o modelo de Árvore de Decisão, que no padrão do sklearn acabou gerando overfitting (decora os dados de treino) e caiu para ~77.1%. O uso da mediana também se mostrou levemente melhor do que a média, já que idades e valores tarifários costumam ter outliers no Titanic, e a mediana não é afetada por esses valores extremos.

###**2. O que acontece se você não escalonar os dados?**

Matematicamente, a Regressão Logística calcula distâncias e tenta otimizar pesos. Sem o escalonador, variáveis como Idade (que vai até 80) têm uma escala muito diferente de Pclass (que é apenas 1, 2 ou 3). O modelo tem mais dificuldade em dar pesos justos a todas as variáveis e demora mais para encontrar a solução ótima (convergência). O escalonamento garante que nenhuma variável "puxe" a importância toda para si apenas porque os seus números são absolutos e maiores. Modelos baseados em árvore não sofrem com isso, mas os baseados em distância, sim.

###**3. O que acontece se você não tratar os valores nulos (vetores nulos)?**

O código de Machine Learning (os algoritmos do Scikit-Learn) quebra e exibe um erro (ValueError: Input contains NaN). Os modelos de Machine Learning matemáticos dependem de matrizes compostas 100% por números válidos. Como há muitos dados faltando na coluna Age (Idade) do Titanic, tentar usar o .fit() passando valores NaN fará o algoritmo falhar imediatamente, pois não é possível fazer multiplicação de matrizes numéricas tendo dados ausentes no meio.